# Notebook 7 — Objective-Driven Autonomous Nanoindentation (Modular Architecture)

Same goal as notebook 5 — state a research objective in plain English and let an
orchestrated agent campaign pursue it with GP/EI Bayesian optimization — rebuilt on the
architecture principle that **the notebook is a control panel, not a codebase**.

```
 notebooks/Notebook_7.ipynb          <- ~25 short cells: config, gates, calls, results
 nanoindentation_agents/
   instrument/                       <- everything that touches the iMicro
     session.py     strict/soft call policy, capture, place ONE array (never moves)
     motion.py      closed-loop jogs, backlash policy, PERSISTED + ENFORCED calibration
     export.py      the InView ReviewData/export contract (family names, .NMD waits)
     screenread.py  live H/E recorder (the safety net while an array runs)
     results.py     parsing + the validation ladder (CSV preferred, never trusted)
   campaign/
     journal.py     append-only JSONL event log -> survives kernel restarts, enables resume
     approval.py    the WHOLE safety model in one file (manual / sim-auto / autonomous ack)
     runner.py      ONE round implementation: propose_round() -> execute_round()
   agents/measurement_strategy_agent.py <- bounded per-site recipe advisor
```

**What this fixes over notebook 5:** the round logic existed in three diverging copies
(Section 12, cells A–D, Section 14) — here round 1, manual rounds, and the autonomous
loop all run the *same* two calls. Campaign state that used to live only in kernel
memory (`RUN_RECORDS`, sample counters, learned jog gains) is journaled to disk, so a
crash mid-campaign no longer orphans data already on the instrument.

**Safety model (explicit, not emergent):**
1. Every gate defaults to **pending** — `MY_DECISION = ""` ships empty; approval is an edit.
2. Nothing auto-approves via environment variables. Simulator auto-approval refuses to
   construct against real hardware; autonomous real-hardware runs require typing an
   acknowledgement phrase **in the session** (see `campaign/approval.py`).
3. On real hardware the runner **refuses to start** without a fresh stage calibration —
   the 8/03 runaway (X axis inverted, 290 → 1247 µm) is why that is a precondition, not
   a convention.
4. Positioning is *best-effort by design*: one honest jog, indent where the stage really
   is, record the live readback. BO needs true (position, measurement) pairs, not the
   exact site it asked for.
5. The Measurement Strategy Agent has no instrument tools. It can only propose values
   for features explicitly set to `agent`; validators and the human gate remain authoritative.


## 1. Objective & switches

**`USER_OBJECTIVE` is the only required input.** Everything below it is an operator
switch with a safe default: no switch in this notebook can place an indent by itself.


In [ ]:
import os, sys, json, time, warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_colwidth", 90)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "nanoindentation_agents").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ----------------------------- THE ONLY REQUIRED INPUT -----------------------------

USER_OBJECTIVE = os.environ.get("NB7_OBJECTIVE",
    "Characterize mechanically accessible visible particulates embedded in the "
    "epoxy mount inside the brass ring while using as few indents as possible. "
    "Segment the optical image into particle regions, reject epoxy and brass-ring "
    "areas, and select only high-confidence indentation targets located well within "
    "particle interiors and away from edges, cracks, scratches, pores, defects, or "
    "overlapping particles. Use a single indent per particle whenever that indent is "
    "sufficient to represent the particle. Apply additional spaced indents only on "
    "large or heterogeneous particles where multiple measurements are necessary to "
    "capture meaningful variation. Prioritize representative coverage over exhausting "
    "the full indent budget, and stop once the minimum necessary set of valid particle "
    "measurements has been collected. Return a particle-by-particle map of hardness "
    "and modulus.")
# ----------------------------- operator switches ------------------------------------
OPERATOR_NAME        = "operator"
USE_SIMULATOR        = False     # False -> real iMicro Flask server
RUN_SIMULATOR_SERVER = False     # start the si mulator from this notebook
# SKIP_SCREENING bypasses the live screen-read monitor (OCR + depth polling,
# per-indent H/E capture) entirely and trusts the instrument's result file the
# moment it appears, with Process Review/Excel as the SOLE data source for every
# array (no screen-read fallback). Applies uniformly to manual rounds (section 7)
# AND the autonomous loop (section 8) via CampaignConfig — one knob, read once
# when CampaignConfig is built below. Operator-accepted tradeoff: on real
# hardware this bare file-exists signal has already fired before a multi-indent
# grid actually finished (see InstrumentSession.place_array's screen-read path);
# with screening off there is nothing left to catch that.
SKIP_SCREENING       = True     # True -> trust Process Review/Excel alone
# HOW LONG ONE INDENT TAKES, in seconds (measured ~2 min 30 s on 2026-08-18).
# InView writes the result file after the FIRST indent, so once it appears the
# monitor holds for (nx*ny - 1) * INDENT_DURATION_S before Process Review runs:
#   1x1 -> 0 s      2x2 -> 450 s (7.5 min)
#   3x3 -> 1200 s (20 min)      5x5 -> 3600 s (60 min)
# Tune this one number if rounds export mid-array (raise it) or waste time on
# the instrument (lower it). The grid size is always read from the plan.
INDENT_DURATION_S    = 70.0
INDENT_BUDGET = int(os.environ.get("NB7_INDENT_BUDGET", "200"))  # rounds are the real budget
MAX_ROUNDS    = int(os.environ.get("NB7_MAX_ROUNDS", "20"))
SAMPLE_PREFIX = os.environ.get("NB7_SAMPLE_PREFIX", "test")        # -> test_01, test_02, ...

# InView sitting: THE one path knob. InView creates the folder; this only points at it,
# and preflight verifies it against the instrument before any indent.
TEST_NUMBER  = int(os.environ.get("NB7_TEST_NUMBER", "1"))
SESSION_DATE = os.environ.get("NB7_SESSION_DATE") or time.strftime("%Y%m%d")

# Which ReviewData export button the campaign clicks: "csv" or "excel".
# "excel" makes InView write .xlsx instead of .CSV; everything downstream
# (locating, downloading, parsing) follows the format automatically, and the
# local reader then needs openpyxl.
EXPORT_FORMAT = os.environ.get("NB7_EXPORT_FORMAT", "excel").strip().lower()

RUN_ID = os.environ.get("NB7_RUN_ID") or time.strftime("%m%d_%H%M%S")

from nanoindentation_agents.notebook_credentials import (
    load_project_env, resolve_hf_token, resolve_instrument_token)
load_project_env(REPO_ROOT)
INSTRUMENT_HOST  = os.environ.get(
    "NANOINDENT_HOST", "127.0.0.1" if USE_SIMULATOR else "YOUR_INSTRUMENT_IP")
INSTRUMENT_PORT  = int(os.environ.get("NANOINDENT_PORT", "5005" if USE_SIMULATOR else "5000"))
INSTRUMENT_TOKEN = resolve_instrument_token()
DISABLE_LLM      = os.environ.get("NB7_DISABLE_LLM", "0") == "1"
TRY_LLM          = bool(resolve_hf_token()) and not DISABLE_LLM
TRY_VISION_LLM   = TRY_LLM

# Per-site recipe advisor: OFF is the behavior-preserving default. When ON, each
# feature below still independently chooses agent authority or its fixed/default value.
ENABLE_MEASUREMENT_STRATEGY_AGENT = True #(
    #os.environ.get("NB7_ENABLE_MEASUREMENT_STRATEGY_AGENT", "0") == "1")
MEASUREMENT_CONTROL_MODES = {
    "site_action": "agent",       # indent or skip this proposed site
    "method_type": "fixed",      # keep the proven EH workflow
    "control_mode": "fixed",     # depth-controlled unless explicitly delegated
    "grid_nx": "agent",
    "grid_ny": "agent",
    "spacing_um": "agent",
    "rotation_deg": "fixed",
    "target depth": "agent",
    "target load": "agent",
    "strain rate": "fixed",
    "surface approach distance": "fixed",  # safety locked
    "surface approach velocity": "fixed",  # safety locked
    "poisson": "fixed",                    # material/operator facts
    "metal": "fixed",
    "drift": "fixed",
}

print(f"Mode: {'SIMULATOR' if USE_SIMULATOR else 'REAL iMicro'} @ "
      f"http://{INSTRUMENT_HOST}:{INSTRUMENT_PORT} | run {RUN_ID} | "
      f"LLM {'on' if TRY_LLM else 'off (deterministic fallbacks)'}")
print(f"Budget: {INDENT_BUDGET} indents / {MAX_ROUNDS} adaptive rounds | operator: {OPERATOR_NAME}")
print(f"InView export format: {EXPORT_FORMAT.upper()}")
print(f"Measurement Strategy Agent: {'ENABLED' if ENABLE_MEASUREMENT_STRATEGY_AGENT else 'OFF (fixed/default fallback)'}")
print("OBJECTIVE:", USER_OBJECTIVE)


## 2. Connect & compose

One cell builds the whole stack: toolkit → session → journal → motion → exporter.
The **journal** is the durable spine — every gate, jog, array, export, and result
appends one JSONL line to `notebooks/campaign_journals/campaign_<RUN_ID>.jsonl`.


In [ ]:
import subprocess
from nanoindentation_agents import notebook_ui as ui
from nanoindentation_agents.state import create_initial_state
from nanoindentation_agents.config import load_config, set_active_config
from nanoindentation_agents.tools.indent_tools import IndentToolkit, set_toolkit
from nanoindentation_agents.instrument import (
    InstrumentSession, MotionController, ExportConfig, ExportPipeline,
    ScreenReadConfig, results as res)
from nanoindentation_agents.campaign import CampaignJournal

SIM_DATA_DIR  = REPO_ROOT / "sim_instrument_data"
DOWNLOAD_DIR  = REPO_ROOT / "notebooks" / "downloaded_data"
ANNOTATED_DIR = REPO_ROOT / "notebooks" / "annotated_images"
JOURNAL_DIR   = REPO_ROOT / "notebooks" / "campaign_journals"

sim_process = globals().get("sim_process")
if USE_SIMULATOR and RUN_SIMULATOR_SERVER:
    if sim_process is None or sim_process.poll() is not None:
        sim_process = subprocess.Popen(
            [sys.executable, str(REPO_ROOT / "scripts" / "run_sim_server.py"),
             "--host", INSTRUMENT_HOST, "--port", str(INSTRUMENT_PORT),
             "--data-dir", str(SIM_DATA_DIR)],
            cwd=str(REPO_ROOT), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        time.sleep(1.5)
        print(f"Simulator server started (PID {sim_process.pid})")

cfg = load_config(None)
cfg.research_question = cfg.experiment_context = USER_OBJECTIVE
cfg.instrument.host, cfg.instrument.port = INSTRUMENT_HOST, INSTRUMENT_PORT
cfg.instrument.token, cfg.instrument.sim_mode = INSTRUMENT_TOKEN, USE_SIMULATOR
cfg.literature.local_dir = str(REPO_ROOT / "literature")
cfg.memory.base_dir = str(REPO_ROOT / "experiment_memory")
cfg.measurement_strategy.enabled = ENABLE_MEASUREMENT_STRATEGY_AGENT
cfg.measurement_strategy.site_action.mode = MEASUREMENT_CONTROL_MODES["site_action"]
cfg.measurement_strategy.method_type.mode = MEASUREMENT_CONTROL_MODES["method_type"]
cfg.indent_defaults.method_control.control_mode_agent_selectable = (
    MEASUREMENT_CONTROL_MODES["control_mode"] == "agent")
for feature, spec in cfg.measurement_strategy.grid_control.params.items():
    spec.mode = MEASUREMENT_CONTROL_MODES[feature]
for feature, spec in cfg.indent_defaults.method_control.params.items():
    spec.mode = MEASUREMENT_CONTROL_MODES[feature]
invalid_modes = {k: v for k, v in MEASUREMENT_CONTROL_MODES.items()
                 if v not in {"fixed", "agent"}}
assert not invalid_modes, f"Unknown control modes: {invalid_modes}"
set_active_config(cfg)

toolkit = IndentToolkit(host=INSTRUMENT_HOST, port=INSTRUMENT_PORT,
                        token=INSTRUMENT_TOKEN, timeout=cfg.instrument.timeout)
set_toolkit(toolkit)
assert toolkit.ping(), "Instrument server unreachable — check section 1 / VPN."

journal  = CampaignJournal(JOURNAL_DIR, RUN_ID)
session  = InstrumentSession(toolkit, download_dir=DOWNLOAD_DIR, run_id=RUN_ID,
                             journal=journal,
                             screen_config=ScreenReadConfig(csv_dir=str(DOWNLOAD_DIR)))
motion   = MotionController(session, use_simulator=USE_SIMULATOR,
                            calibration_path=JOURNAL_DIR / "stage_calibration.json")
exporter = ExportPipeline(session, ExportConfig(test_number=TEST_NUMBER,
                                                session_date=SESSION_DATE,
                                                export_format=EXPORT_FORMAT,
                                                use_simulator=USE_SIMULATOR),
                          download_dir=DOWNLOAD_DIR)

state = create_initial_state(research_question=USER_OBJECTIVE,
                             experiment_name="nb7_objective_campaign",
                             max_iterations=cfg.experiment.max_iterations,
                             ic_max_cycles=cfg.experiment.ic_max_cycles)
msg_log = ui.AgentMessageLog()
ui.ApprovalGate.registry.clear()

print(f"Connection OK | journal -> {journal.path}")
print(f"Motion: {motion.summary()}")


## 3. 🎯 Compile the objective — Gate 0

The orchestrator translates the free-text goal into a structured `ResearchObjective`
(LLM with a validated strict-JSON prompt when available, deterministic parser
otherwise). **You approve the machine's understanding before anything else runs.**

The campaign-shaping block is the one place experiment geometry is overridden — it is
visible here, on purpose, rather than buried in library defaults.


In [ ]:
from nanoindentation_agents.objective import compile_objective
from nanoindentation_agents.agents.orchestrator_agent import OrchestratorAgent

objective = compile_objective(USER_OBJECTIVE, cfg=cfg,
                              fov_bounds=(0.0, 100.0, 0.0, 75.0),  # re-locked after Observe
                              indent_budget=INDENT_BUDGET, max_rounds=MAX_ROUNDS,
                              use_llm=TRY_LLM)

# ---- campaign shaping (explicit overrides): rounds are the budget -------------------
# The 2x2 geometry below is the fixed/default fallback. If grid features are delegated
# and the strategy agent is enabled, it may select a smaller validated grid per site.
# Success criteria stay pinned to what MAX_ROUNDS can deliver.
objective.constraints.grid_nx = objective.constraints.grid_ny = 2
objective.constraints.spacing_um = 15
objective.constraints.indent_budget = INDENT_BUDGET
objective.success.min_visited_locations = max(
    1, min(objective.success.min_visited_locations, MAX_ROUNDS - 1))
objective.success.min_confirming_arrays = max(
    1, min(objective.success.min_confirming_arrays, MAX_ROUNDS - 2))

orch = OrchestratorAgent(objective, cfg, use_llm=TRY_LLM)
state.update(orch.objective_state_updates())

display(pd.DataFrame([{"property": g.name, "direction": g.direction, "weight": g.weight,
                       "target range (GPa)": str(g.target_range) if g.target_range else "-"}
                      for g in objective.property_goals]))
print(f"Compiled by: {objective.compiled_by.upper()} | material: "
      f"{objective.material_system or '(not stated)'}")
print(f"Budget: {objective.constraints.indent_budget} indents | "
      f"{objective.constraints.max_rounds} rounds | "
      f"{objective.constraints.grid_nx}x{objective.constraints.grid_ny} arrays "
      f"@ {objective.constraints.spacing_um:g} um")
print(f"Success: >= {objective.success.min_improvement_pct:g}% over baseline, "
      f">= {objective.success.min_confirming_arrays} arrays, "
      f">= {objective.success.min_visited_locations} locations")

objective_gate = ui.ApprovalGate(
    "gate-0-objective", "Research objective approval",
    description=("Approve the orchestrator's interpretation of your research goal. "
                 "All briefings, success criteria and the budget derive from it.\n\n"
                 + objective.summary_text()))
msg_log.log("Orchestrator Agent", "Human Operator",
            f"Objective compiled ({objective.compiled_by}) — awaiting Gate 0",
            detail=objective.summary_text()[:300])


In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE — the gate ships EMPTY on purpose. <==
#     MY_DECISION = "approve"            (optionally "approve: note")
#     MY_DECISION = "reject: <reason>"   (then refine USER_OBJECTIVE and re-run §3)
MY_DECISION = "approve"

from nanoindentation_agents.campaign import apply_decision
if apply_decision(objective_gate, MY_DECISION, OPERATOR_NAME):
    journal.append("GATE", gate_id="gate-0-objective",
                   approved=objective_gate.is_approved, by=OPERATOR_NAME)
    msg_log.log("Human Operator", "Orchestrator Agent",
                "Objective APPROVED — brief the sub-agents" if objective_gate.is_approved
                else "Objective REJECTED — refine USER_OBJECTIVE and re-run section 3")


In [ ]:
objective_gate.require_approved()
state.update(orch.briefing_state_updates(state))
display(pd.DataFrame([{"agent role": r, "orchestrator directive": t}
                      for r, t in state["orchestrator_briefings"].items()]))
print(f"Literature search pre-seeded: {state['literature_search_query']!r}")

from nanoindentation_agents.campaign import CampaignConfig, CampaignRunner
campaign = CampaignConfig(operator=OPERATOR_NAME, use_simulator=USE_SIMULATOR,
                          run_id=RUN_ID, sample_prefix=SAMPLE_PREFIX,
                          use_vision_llm=TRY_VISION_LLM, skip_screening=SKIP_SCREENING,
                          use_measurement_strategy_llm=TRY_LLM,
                          indent_duration_s=INDENT_DURATION_S,
                          annotated_dir=str(ANNOTATED_DIR))
runner = CampaignRunner(session=session, motion=motion, exporter=exporter,
                        journal=journal, objective=objective, orchestrator=orch,
                        state=state, app_cfg=cfg, campaign=campaign, msg_log=msg_log)
journal.append("CAMPAIGN_START", objective=USER_OBJECTIVE, operator=OPERATOR_NAME,
               simulator=USE_SIMULATOR, budget=INDENT_BUDGET, max_rounds=MAX_ROUNDS,
               measurement_strategy_enabled=ENABLE_MEASUREMENT_STRATEGY_AGENT,
               measurement_control_modes=MEASUREMENT_CONTROL_MODES)
print("Runner composed — the same propose/execute path serves manual AND autonomous rounds.")
print(f"Measurement Strategy Agent: {'ACTIVE' if ENABLE_MEASUREMENT_STRATEGY_AGENT and TRY_LLM else 'fixed/default fallback'}")
display(pd.DataFrame(runner.measurement_strategy.control_contract_rows()))
print(f"skip_screening: {SKIP_SCREENING} "
      f"({'Process Review/Excel is the sole data source' if SKIP_SCREENING else 'screen-read monitor active'})")
print(f"indent duration estimate: {INDENT_DURATION_S:.0f}s -> a 2x2 waits "
      f"{3 * INDENT_DURATION_S / 60:.1f} min after the result file appears")


## 4. 🔭 Observe, lock the search area, rank targets

The first capture fixes the campaign's coordinate frame. On real hardware the FOV is
**calibrated from the burned-in scale bar** (hard stop if unreadable — never move an
indenter on an unknown scale); the ranked targets seed round 1.


In [ ]:
from nanoindentation_agents.agents.observe_agent import observe_node

state.update(observe_node(state))
runner.lock_search_area()
ui.show_capture(state["current_image"], title="Initial video panel (search area)",
                note="This frame defines the campaign's search area.",
                fov_width_um=campaign.fov_w_um, fov_height_um=campaign.fov_h_um)

ranked = runner.analyze_current_frame(0, tag="initial", refresh_plan=True)
display(pd.DataFrame([{"rank": t["priority"], "target": t.get("label", "?"),
                       "x (um)": round(t.get("x_um", 0), 1),
                       "y (um)": round(t.get("y_um", 0), 1),
                       "objective score": t.get("objective_score"),
                       "why": t.get("rank_reason", "")}
                      for t in ranked[:6]]))


## 5. 📚💡 Context agents (optional) — Gate 1

Literature and hypothesis give the campaign scientific context and an auditable
prediction; they do not steer the acquisition function. Skip with
`RUN_CONTEXT_AGENTS = False` — the BO campaign is unaffected.


In [ ]:
RUN_CONTEXT_AGENTS = True

if RUN_CONTEXT_AGENTS:
    from nanoindentation_agents.agents.literature_agent import literature_node
    from nanoindentation_agents.agents.hypothesis_agent import hypothesis_node
    from nanoindentation_agents.critics.hypothesis_critic import hypothesis_critic_node

    lit = ui.run_with_timeout(lambda: literature_node(state), seconds=240,
                              label="Literature Agent")
    if lit and (lit.get("literature_context") or "").strip():
        state.update(lit)
    else:
        from nanoindentation_agents.rag.literature_retriever import LiteratureRetriever
        ctx, srcs = LiteratureRetriever(cfg.literature).retrieve(
            state["literature_search_query"],
            keywords=state.get("literature_search_keywords") or ["hardness", "modulus"])
        state.update({"literature_context": ctx[:2500], "literature_raw_context": ctx,
                      "literature_sources": srcs,
                      "literature_search_meta": {"mode": "local_files (offline fallback)"}})
    ui.literature_panel(state)

    hyp = ui.run_with_timeout(lambda: hypothesis_node(state), seconds=240,
                              label="Hypothesis Agent")
    if not hyp or not (hyp.get("hypothesis") or "").strip():
        goal = "; ".join(g.describe() for g in objective.property_goals)
        pct = objective.success.min_improvement_pct
        text = (f"We hypothesize that distinct microstructural regions of the "
                f"{objective.material_system or 'sample'} differ measurably in the "
                f"objective properties ({goal}); the objective is met if the best site "
                f"beats the sample baseline by >= {pct:g}% on the weighted score, and "
                f"spatially uniform properties refute it.")
        hyp = {"hypothesis": text,
               "hypothesis_history": state["hypothesis_history"] + [text],
               "hypothesis_data": {"hypothesis": text,
                                   "success_criteria": f">= {pct:g}% above baseline",
                                   "failure_criteria": "no site beats baseline in budget"}}
    state.update(hyp)
    ui.hypothesis_panel(state)
    for _ in range(3):
        state.update(hypothesis_critic_node(state))
        ui.critic_panel(f"Hypothesis Critic — pass {state['hypothesis_critic_passes']}",
                        state["hypothesis_critique"], state["hypothesis_critique_approved"])
        if (state["hypothesis_critique_approved"]
                and state["hypothesis_critic_passes"]
                >= cfg.experiment.min_hypothesis_critic_passes):
            break
        upd = ui.run_with_timeout(lambda: hypothesis_node(state), seconds=240,
                                  label="Hypothesis Agent (revision)")
        if upd:
            state.update(upd)
else:
    print("Context agents skipped.")

hypothesis_gate = ui.ApprovalGate(
    "gate-1-hypothesis", "Hypothesis / context sign-off",
    description="Approve to continue to calibration and rounds. No stage motion or "
                "indentation is possible until later gates are also approved.")


In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE <==   ("approve" / "reject: <reason>")
MY_DECISION = "approve"

if apply_decision(hypothesis_gate, MY_DECISION, OPERATOR_NAME):
    state["user_approved"] = hypothesis_gate.is_approved
    journal.append("GATE", gate_id="gate-1-hypothesis",
                   approved=hypothesis_gate.is_approved, by=OPERATOR_NAME)


## 6. 🛠 Stage calibration & preflight — enforced, not suggested

`motion.calibrate()` measures, per axis: which arrow moves which way, the true gain,
and what a reversal costs (6 small jogs, **no indents**, ~2 min). It persists to
`campaign_journals/stage_calibration.json`; **`runner.preflight()` refuses to start a
real-hardware campaign without a calibration fresher than
`campaign.calibration_max_age_h`** (default 12 h).

**The measurements have been contradictory, and that contradiction is the important
fact.** An isolated bench test (nothing else running) measured a clean stage: X inverted,
gain exactly 1.00, zero cross-axis coupling. Two live `calibrate()` runs *inside* a full
campaign session measured something very different: gain up to 5×, and — critically —
**85–169 µm of movement on one axis from a command aimed at the other axis.** Cross-talk
that large on a single-axis command isn't a stage property; it's the signature of clicks
landing on the wrong on-screen control because the InView window has drifted off the
screen position the click coordinates assume.

`motion.calibrate()` now checks this first (`window_check()`, via the server's
`report_window_rects`) and refuses to spend probe jogs if the window is out of place —
bring it back to its calibrated screen position and retry. See `CALIBRATION_MODE` in the
next cell for a "manual" bypass that adopts the isolated bench-test numbers directly
instead of re-probing.

In [ ]:
hypothesis_gate.require_approved()

# ==============================================================================
# CALIBRATION_MODE — how motion.cfg gets its dir_x/dir_y/gain_seed this session.
#
#   "auto"   (default) — motion.calibrate() probes the stage live: 6 small jogs,
#            NO indents. Each probe move is measured ON ITS OWN
#            (move -> SETTLE -> read), so a single bad move is visible instead
#            of being averaged into a two-move total. The probe now REFUSES to
#            save a result it cannot defend — repeat jogs that disagree on sign
#            or gain, an axis that barely moves, or cross-axis motion from a
#            single-axis command all raise instead of being written to disk.
#            Earlier versions fitted numbers to whatever they observed and saved
#            them with a confident fresh timestamp; that is how a BACKWARDS X
#            sign reached the stage three times on 8/03-8/04. Wrong is worse
#            than missing, because missing is obvious.
#
#            The window check runs first when the server provides
#            report_window_rects. If that endpoint is NOT deployed the probe
#            still runs and the result is tagged window_verified=false — "I
#            cannot tell where the window is" is missing information, not
#            evidence of danger, and must not brick calibration.
#
#   "manual" — BYPASS the live probe; adopt the BENCH_* values below. Use when
#            you trust a specific external measurement more than the in-session
#            probe. Does NOT disable the runtime net: jog_to() still raises if
#            the first live jog responds opposite to what was pinned.
#
#   "skip"   — bypass calibration ENFORCEMENT entirely. Troubleshooting only.
#
# SETTLE_S IS THE KNOB THAT MATTERED. The move tool's `t` only paces the clicks
# INSIDE InView's Relative Move dialog and returns BEFORE the stage has arrived,
# so a readback taken immediately afterwards can catch the stage mid-flight —
# which is indistinguishable, in the numbers alone, from a wrong axis map. The
# isolated bench protocol that measured cleanly (gain 1.00, zero cross-talk,
# X sign -1) slept 3 s before every readback; calibrate() slept 0 and measured
# the X sign backwards every time. It now settles here and in every jog.
# ==============================================================================
CALIBRATION_MODE = "skip"        # "auto" | "manual" | "skip"
RECALIBRATE      = True          # "auto" only: force a fresh probe even if a
                                 # recent one is on file
SETTLE_S         = 3.0           # seconds to wait after EVERY move before reading
                                 # the position back (real hardware only)

# Trusted values for "manual" mode — isolated bench test 2026-08-03, nothing else
# running: 3 static reads identical to 0.01 um, capture did not move the stage,
# every direction moved EXACTLY the amount asked, zero cross-talk, fully reversible.
BENCH_DIR_X, BENCH_DIR_Y = -1, +1
BENCH_GAIN_X, BENCH_GAIN_Y = 1.0, 1.0

if not USE_SIMULATOR:
    motion.cfg.settle_s = SETTLE_S
    print(f"settle: waiting {motion.cfg.settle_s:g}s after every move before readback")

if USE_SIMULATOR:
    print("Simulator — ideal stage; calibration not required.")
elif CALIBRATION_MODE == "skip":
    campaign.require_motion_calibration = False
    print("WARNING: CALIBRATION_MODE='skip' — enforcement disabled. Using "
          f"whatever is already loaded: {motion.summary()}")
elif CALIBRATION_MODE == "manual":
    motion.pin_calibration(BENCH_DIR_X, BENCH_DIR_Y, BENCH_GAIN_X, BENCH_GAIN_Y,
                           source="isolated bench test 2026-08-03",
                           note="right/left/up/down 20 um each moved exactly 20 um "
                                "on one axis, 0.00 um cross-talk, fully reversible")
    journal.append("CALIBRATION", **{k: v for k, v in motion.calibration.items()
                                     if k != "axes"})
elif (not RECALIBRATE and motion.calibration
      and motion.calibration_age_h <= campaign.calibration_max_age_h):
    motion.apply_saved_calibration()
    print(f"Using saved calibration ({motion.calibration_age_h:.1f} h old): "
          f"{motion.summary()}")
    print("Set RECALIBRATE = True after anyone touches the InView window.")
else:
    # Raises StageSafetyError rather than saving a calibration it cannot defend.
    # If it does raise, read WHICH check failed: raising SETTLE_S is the first
    # thing to try, then re-run with the InView window frontmost, then fall back
    # to CALIBRATION_MODE = "manual".
    motion.calibrate()
    journal.append("CALIBRATION", **{k: v for k, v in motion.calibration.items()
                                     if k != "axes"})

# ---- IMAGE -> STAGE conversion (the OTHER half of "move to the chosen site") ----
# Turning an offset in the video panel into a stage move. BOTH signs here were
# assumptions written as comments until 2026-08-04, and the X one was BACKWARDS:
# a target 56 um LEFT of centre was driven further LEFT until it left the field
# entirely (error 56 -> 111 um) — which is what you saw happen on the instrument.
# Y's sign was right but only 72% of the needed size. Measured:
#     stage +60.0 um X -> scene +58.3 um (RIGHT) => site_gain_x = -1.03
#     stage +60.1 um Y -> scene +43.5 um (DOWN)  => site_gain_y = -1.38
# RE-CHECKED 2026-08-05 against the round-1 frames of run 0805_114613 — a real
# campaign move (+49.2, -24.4) um, not a probe: the scene moved (+47.7, -23.1) um,
# giving site_gain_x = -1.03, site_gain_y = -1.06 at 0.1815 / 0.1861 um/px. The
# SIGNS are confirmed and the scale is very nearly ISOTROPIC, so the earlier
# -1.38 / "38% anisotropy" reading was the badly-determined Y registration that
# imagecal's docstring warns about, not a property of the optics. 1.0 stands.
# RE-MEASURE after any objective / zoom / camera change.
CALIBRATE_IMAGE_TO_STAGE = False   # True -> measure now: XY only, NO indents, ~1 min

if not USE_SIMULATOR:
    if CALIBRATE_IMAGE_TO_STAGE:
        runner.calibrate_image_to_stage(probe_um=60.0)
    else:
        print(f"image->stage: site_gain_x={campaign.site_gain_x:+.2f}, "
              f"site_gain_y={campaign.site_gain_y:+.2f}  (measured 2026-08-04)")
        print("  set CALIBRATE_IMAGE_TO_STAGE = True to re-measure on this field")


# ---- PRE-INDENT ARRIVAL CHECK (verification in IMAGE space) ------------------
# jog_to() proves the stage reached the COORDINATE it was handed — 0.07 um on
# 8/05. It cannot prove that coordinate was the RIGHT one, because it checks the
# stage against the same map that computed the destination. A wrong gain, or a
# wrong SIGN, therefore converges perfectly onto a site ~2x the offset away and
# reports success; on 8/05 the stage log read "0.07 um from target" while the
# planning overlay drew the site 98 um from where it really was.
#
# This closes the loop where it actually matters: at the armed position, the
# live frame is registered against the frame the target was chosen from, and the
# round indents only if the chosen feature is genuinely under the indenter
# crosshair. Out of tolerance -> correct and re-measure -> still out -> STOP.
# A field too flat to register WARNS and proceeds; a check that cannot be
# satisfied is a check that gets switched off.
VERIFY_ARRIVAL   = "enforce"   # "enforce" | "warn" | "off"   (never on the simulator)
ARRIVAL_TOL_UM   = 8.0         # how close the feature must be to the indenter axis
MAX_VISUAL_FIXES = 1           # residual corrections allowed before refusing

campaign.verify_arrival = VERIFY_ARRIVAL
campaign.arrival_tol_um = ARRIVAL_TOL_UM
campaign.max_visual_corrections = MAX_VISUAL_FIXES
if not USE_SIMULATOR:
    print(f"arrival check: {VERIFY_ARRIVAL} — tolerance {ARRIVAL_TOL_UM:g} um, "
          f"{MAX_VISUAL_FIXES} correction(s) allowed")
    print(f"  aiming at the burned-in indenter crosshair: {campaign.aim_at_crosshair} "
          "(it sat 2.9 um above the frame centre on 8/05)")

runner.preflight()              # ping + sitting folder + calibration enforcement
# Optional sanity check that the live screen keys match the recorder's expectations:
# session.screen_read_diagnostic()


## 7. 🎯 Rounds — one implementation, two calls

| call | what happens | instrument |
|---|---|---|
| `runner.propose_round()` | orchestrator decides → fresh frame + analysis → sites → Measurement Strategy Agent resolves/logs each recipe → gate opens | capture only |
| `runner.execute_round(p, decision=...)` | jog (once, honestly) → **indent** → wait for `.NMD` → export/download → validation ladder → record + critic | **irreversible** |

Round 1 and every adaptive round use this same path — there is no separate "round 1"
code to drift out of sync. Repeat the two cells below once per round; `propose_round()`
is idempotent (re-running replaces an undecided gate, touches nothing else).
The proposal table and approval cards identify every agent-decided feature and its final
validated value. On real hardware, agent-changed recipes require this per-round human gate.
If the orchestrator returns `objective_met` / `budget_exhausted` / `escalate_to_human`,
skip to Section 9.


In [ ]:
proposal = runner.propose_round()
if proposal.terminal:
    print(">> Campaign is over — skip to Section 9 (results).")
elif not proposal.plans:
    print(">> No viable sites this round — re-run propose_round() or widen the search.")
else:
    print(f">> Review the gate above, then set MY_DECISION in the next cell "
          f"(round {proposal.round_no}, {len(proposal.plans)} site(s)).")


In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE — indentation is IRREVERSIBLE. <==
#     ("approve" / "reject: <reason>")
MY_DECISION = "approve"

round_results = runner.execute_round(proposal, decision=MY_DECISION)


## 8. 🤖 Autonomous campaign (explicit opt-in)

The loop below is *literally* the manual path in a `while` loop — same positioning
rules, same export waits, same validation, same journal. What changes is only the
approval policy:

* **Simulator** — `AutoApproval(use_simulator=True)` (refuses to construct on hardware).
* **Real hardware** — `AutonomousHardwareApproval` requires typing the acknowledgement
  phrase in this session. That acknowledgement authorizes validated fixed and
  agent-selected recipes for the autonomous run. No environment variable can switch
  it on, and this cell does nothing on a bare Run All.


In [ ]:
from nanoindentation_agents.campaign import (AutoApproval, AutonomousHardwareApproval,
                                             AUTONOMOUS_ACK)

RUN_AUTONOMOUS = True    # set True AND construct a policy below to run hands-free

policy = None
if RUN_AUTONOMOUS and USE_SIMULATOR:
    policy = AutoApproval(use_simulator=True)
elif RUN_AUTONOMOUS:
    # Deliberate friction for real hardware — uncomment BOTH lines to accept that every
    # remaining round, including validated agent-selected recipes, places irreversible
    # indents with no per-round human gate:
    I_UNDERSTAND = AUTONOMOUS_ACK
    policy = AutonomousHardwareApproval(OPERATOR_NAME, ack=I_UNDERSTAND)
    pass

if policy is None:
    print("Autonomous mode not armed — use the manual round cells in Section 7.")
    if RUN_AUTONOMOUS and not USE_SIMULATOR:
        print("(Real hardware: uncomment the acknowledgement lines above to arm it.)")
else:
    stop_reason = runner.run(policy)
    print(f"Stop reason: {stop_reason}")


In [ ]:
# ---- Recovery: re-export every round that fell back to screen data. ----------------
# Places NO indents, moves nothing, safe to re-run. Works even after a kernel restart
# (run records are rebuilt from the journal).
#if any(d.get("needs_recovery") for d in state["indent_history"]):
#    runner.recover_pending(wait_s=60)
#else:
#    print("Nothing to recover — every round has validated export data "
#          "(or no rounds have run yet).")


## 9. 📊 Results — maps, GP posterior, objective attainment


In [ ]:
all_rows = runner.results_rows()
print(f"{len(all_rows)} indents across "
      f"{all_rows['array'].nunique() if len(all_rows) else 0} arrays")
if len(all_rows):
    ui.plot_property_map(all_rows)
    summary = (all_rows.groupby("region")
               .agg(**{"n indents": ("Indent", "count"),
                       "H mean (GPa)": ("Hardness (GPa)", "mean"),
                       "H std": ("Hardness (GPa)", "std"),
                       "E mean (GPa)": ("Modulus (GPa)", "mean"),
                       "E std": ("Modulus (GPa)", "std")}).round(2))
    display(summary)


In [ ]:
import matplotlib.pyplot as plt
from nanoindentation_agents.analysis.objective_acquisition import ObjectiveSurrogate

final_surrogate = ObjectiveSurrogate(objective).fit(state["indent_history"])
final_progress = orch.evaluate_progress(state, final_surrogate)

if final_surrogate.n_points == 0:
    print("NOTE: the surrogate has NO observations — the panels below are the "
          "untrained prior. Fix the data first (recover_pending).")
maps = final_surrogate.posterior_maps(n=70)
best = final_surrogate.best_observed()
fig, axes = plt.subplots(2, 2, figsize=(12.5, 9))
for ax, (key, title, cmap) in zip(axes.ravel(), [
        ("hardness_mean", "predicted hardness (GPa)", "inferno"),
        ("modulus_mean", "predicted modulus (GPa)", "viridis"),
        ("scalar_mean", "objective score (weighted)", "cividis"),
        ("scalar_sigma", "objective uncertainty (sigma)", "magma")]):
    im = ax.imshow(maps[key], origin="lower", extent=maps["extent"],
                   aspect="equal", cmap=cmap)
    ax.scatter(final_surrogate.X[:, 0], final_surrogate.X[:, 1], s=12, c="white",
               edgecolor="k", linewidths=0.4)
    if best:
        ax.scatter([best["x_um"]], [best["y_um"]], marker="*", s=340, c="red",
                   edgecolor="k", linewidths=0.7)
    ax.set_title(title); ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle(f"Final GP posterior ({final_surrogate.mode}, "
             f"{final_surrogate.n_points} indents) — star = best site", y=1.0)
plt.tight_layout(); plt.show()

display(final_progress.table())
if best:
    print(f"Best site: ({best['x_um']:.1f}, {best['y_um']:.1f}) um — "
          f"H={best['hardness_gpa']:.2f} GPa, E={best['modulus_gpa']:.1f} GPa")
print(f"Improvement over baseline: {final_progress.improvement_pct:.1f}% "
      f"(required {objective.success.min_improvement_pct:g}%)")
verdict = "OBJECTIVE MET" if final_progress.all_criteria_met else "OBJECTIVE NOT (YET) MET"
print(f"\n{'=' * 72}\n{verdict}\n{'=' * 72}")


## 9B. 🧑‍🔬 Final Analysis Agent — expert review of the whole campaign

Runs once, after the rounds are done. The numbers are computed deterministically from
the recorded indents and are authoritative; the LLM (role `final_analysis`, falling back
to the default model) only narrates the numbers it is handed — main findings,
cross-test comparison, conclusion, limitations, next steps. Without a token it assembles
the same narrative from the same facts, deterministically.

No instrument calls, no stage motion, no indents: safe to re-run, and safe after
`resume_from_journal()`.

**It refuses to run on screen-read data by default.** A round that fell back to OCR is a
progress signal, not a measurement, and an expert-sounding conclusion built on one reads
as authoritative when it is not. Recover first (`runner.recover_pending()`); or set
`REQUIRE_VALIDATED_DATA = False` to proceed, in which case the affected rounds are named
as provisional in the report's Limitations section rather than passed over in silence.


In [ ]:
# ============ FINAL ANALYSIS AGENT — expert review of every completed test ============
# Reads only what the campaign already recorded: no instrument calls, no stage motion,
# no indents. Safe to re-run, and safe after a resume_from_journal().
from collections import Counter
from nanoindentation_agents.agents.final_analysis_agent import FinalAnalysisAgent

# Hardness bands (GPa) to score each test against, keyed by the region labels in the
# Section 9 summary table (NB7 labels a 1x1 round by its site, e.g. "bo_round3_site1").
# Leave empty unless you have a defensible reference: "no reference" reads better in the
# report than a band this sample was never meant to match.
LIT_EXPECTATIONS = globals().get("LIT_EXPECTATIONS", {})

# A conclusion is only as good as the data source under it. Screen-read numbers are a
# fallback, not a measurement — recover them before anything gets written down.
REQUIRE_VALIDATED_DATA = True

all_rows = runner.results_rows()
pending  = [d["sample_name"] for d in state["indent_history"] if d.get("needs_recovery")]
print("data source per round:",
      dict(Counter(d.get("data_source", "unknown") for d in state["indent_history"])))

if len(all_rows) == 0:
    print("No indents recorded — nothing to analyze.")
elif pending and REQUIRE_VALIDATED_DATA:
    print(f"BLOCKED: {len(pending)} round(s) still on screen data: {pending}\n"
          "  -> run the Section 8 recovery cell (runner.recover_pending(); no new indents),\n"
          "     then re-run this cell. Set REQUIRE_VALIDATED_DATA = False to analyze anyway.")
else:
    ui.agent_banner("Final Analysis Agent",
                    "reviewing every completed test like a nanoindentation expert")

    # Section 9 computes final_progress; recompute it if that cell was skipped.
    final_progress = globals().get("final_progress")
    if final_progress is None:
        from nanoindentation_agents.analysis.objective_acquisition import ObjectiveSurrogate
        final_progress = orch.evaluate_progress(
            state, ObjectiveSurrogate(objective).fit(state["indent_history"]))

    analyst = FinalAnalysisAgent(objective, cfg, use_llm=TRY_LLM)
    report  = analyst.analyze(all_rows, state=state,
                              expectations=LIT_EXPECTATIONS, progress=final_progress)

    if pending:   # only reachable with REQUIRE_VALIDATED_DATA = False — say so in the report
        note = (f"{len(pending)} round(s) were read off the instrument screen rather than a "
                f"validated export file ({', '.join(pending)}) — treat those numbers as "
                f"provisional.")
        report.limitations = f"{report.limitations}\n\n{note}".strip()
        report.flags.append(note)

    report.display()

    # Persist it where this notebook keeps the rest of its evidence.
    state["final_analysis"] = report.to_dict()
    journal.append("FINAL_ANALYSIS", advisor=report.advisor, n_indents=report.n_indents,
                   n_tests=report.n_tests, headline=report.headline, flags=report.flags,
                   conclusion=report.conclusion[:600])
    md_path = REPO_ROOT / "experiment_logs" / f"nb7_final_analysis_{RUN_ID}.md"
    md_path.parent.mkdir(parents=True, exist_ok=True)
    md_path.write_text(report.to_markdown(), encoding="utf-8")
    msg_log.log("Final Analysis Agent", "Orchestrator Agent",
                "Campaign review complete — findings, conclusion, limitations, next steps.",
                detail=report.conclusion[:280])
    print(f"\nNarrative by: {report.advisor.upper()}  |  saved: {md_path}")


## 10. 🧾 Run report & audit

The journal **is** the audit trail; the JSON report is a rendering of it plus the
compiled objective, decisions, and gate records. The first strategy table summarizes
each target recipe; the second shows every feature's mode, requested value, final
validated value, decision owner, adjustment flag, and reason.


In [ ]:
print(json.dumps(journal.summary(), indent=2))
display(journal.measurement_strategy_history())
display(journal.measurement_strategy_feature_history())
display(journal.method_plan_history())
display(ui.ApprovalGate.audit_table())
msg_log.render_transcript()

report_path = ui.save_run_report(
    REPO_ROOT / "experiment_logs" / f"nb7_campaign_{RUN_ID}.json",
    operator=OPERATOR_NAME, user_objective=USER_OBJECTIVE,
    objective=state.get("research_objective"),
    orchestrator_briefings=state.get("orchestrator_briefings"),
    orchestrator_decisions=state.get("orchestrator_decisions", []),
    measurement_strategy_enabled=ENABLE_MEASUREMENT_STRATEGY_AGENT,
    measurement_control_modes=MEASUREMENT_CONTROL_MODES,
    measurement_strategy_decisions=state.get("measurement_strategy_decisions", []),
    objective_progress=final_progress.to_dict(),
    acquisition=state.get("acquisition_meta", {}),
    hypothesis=state.get("hypothesis"),
    literature={"query": state.get("literature_search_query"),
                "sources": state.get("literature_sources")},
    indent_arrays=[{k: v for k, v in d.items() if k != "results"}
                   for d in state["indent_history"]],
    measurement_critique=state.get("measurement_critique"),
    final_decision=state.get("measurement_critique_decision"),
    final_analysis=state.get("final_analysis"),   # from Section 9B, if it ran
    approval_gates=[g.record() for g in ui.ApprovalGate.registry],
    agent_messages=msg_log.entries,
    campaign_journal=str(journal.path))
print("Report saved:", report_path)


## 11. ♻️ Resume after a kernel restart / shutdown

The journal makes a crash recoverable: re-run sections **1–3** (same `TEST_NUMBER`;
approve Gate 0), then point a journal at the old run id and rebuild. Physical indents
whose exports never validated are then re-exported with **no new indents**.

```python
journal = CampaignJournal(JOURNAL_DIR, "<old RUN_ID>")     # see the table below
# rebuild session/runner against that journal (re-run the compose + runner cells),
runner.resume_from_journal()      # indent history + budget + GP observations
runner.recover_pending()          # re-export anything still on screen data
```


In [ ]:
runs = CampaignJournal.list_runs(JOURNAL_DIR)
display(pd.DataFrame(runs)) if runs else print("No journaled runs yet.")

if sim_process is not None and sim_process.poll() is None:
    sim_process.terminate()
    try:
        sim_process.wait(timeout=5)
    except Exception:
        sim_process.kill()
    print("Simulator server stopped.")


## 12. 📄 Publication figures — the whole set, in one call

Ten journal-ready figures from the finished campaign. Run this **after** the rounds are
done (and after `recover_pending()`, so nothing is plotted from screen-read numbers).
It touches no instrument: it reads the journal, the downloaded InView CSVs, and the
saved frames.

| # | figure | what it is evidence for |
|---|---|---|
| 1 | sites measured, on the sample | where the agent chose to look |
| 2 | hardness & modulus per test | the measurement itself |
| 3 | load–displacement curves | the raw data behind every number |
| 4 | depth-resolved H and E | indentation size effect / tip-area validity |
| 5 | GP posterior + uncertainty | what the agent believed, and where it was unsure |
| 6 | improvement vs baseline by round | that it learned |
| 7 | expected improvement by round | that it converged |
| 8 | hardness against modulus | the material relationship |
| 9 | positioning & arrival accuracy | that it hit what it aimed at |
| 10 | wall-clock anatomy of a round | that it ran unattended |

**House style** (`PubStyle`): 8 pt sans, 0.6 pt axes, ticks in, no top/right spines,
TrueType-embedded vector PDF **plus** 600 dpi PNG. Colours are not hand-picked — the
categorical order clears colour-vision-deficiency separation ΔE 15.8 (target 8) on a
white surface, rounds take an ordinal one-hue ramp because rounds are ordered, and no
figure uses two y-axes. See the module docstring for the validation record.

A figure whose inputs are missing is **skipped and recorded as skipped** in
`figures_manifest.json` rather than drawn half-empty. Draft captions, with the numbers
filled in from the data, are written to `captions.md` next to the figures.

Re-plotting an old run later needs no notebook state:

```python
CampaignFigures.from_journal(JOURNAL_DIR, "0806_131828",
                             download_dir=DOWNLOAD_DIR).generate_all(out)
```


In [ ]:
# ============ PUBLICATION FIGURES — generate the whole set ============
# Reads recorded artefacts only: no instrument calls, no stage motion, no indents.
# Safe to re-run, and safe after a kernel restart (see from_journal below).
from nanoindentation_agents.analysis.publication import CampaignFigures, PubStyle

FIG_DIR = REPO_ROOT / "paper" / "figures" / RUN_ID
FORMATS = ("pdf", "png")          # PDF (vector, TrueType) for the journal; PNG for talks
FIG_STYLE = PubStyle(base_pt=8, dpi=600)

# ---- Figure 1a: a real micrograph, if you have one --------------------------------
# OFF by default and that is deliberate. The frames this pipeline saves are screenshots
# of the whole InView window — toolbar, focus widget, InView's own scale bar and its
# "Single Engage" banner included — so publishing one as "the sample surface" would put
# instrument chrome in a journal figure. Point MICROGRAPH at a real image, and use
# MICROGRAPH_CROP = (left, top, right, bottom) in pixels to trim a saved frame down to
# the video panel if that is all you have.
MICROGRAPH = None
MICROGRAPH_CROP = None

# The GP panels need a fitted surrogate; Section 9 makes one, and this refits if it
# does not exist yet (e.g. you jumped straight here).
if "final_surrogate" not in globals():
    from nanoindentation_agents.analysis.objective_acquisition import ObjectiveSurrogate
    final_surrogate = ObjectiveSurrogate(objective).fit(state["indent_history"])
    final_progress = orch.evaluate_progress(state, final_surrogate)

pending = [d["sample_name"] for d in state["indent_history"] if d.get("needs_recovery")]
if pending:
    print(f"NOTE: {len(pending)} round(s) still on screen-read data {pending} — those "
          "points are plotted from OCR, not from a validated export. Run "
          "runner.recover_pending() first if you can.\n")

figs = CampaignFigures.from_runner(runner, state=state, surrogate=final_surrogate,
                                   progress=final_progress, style=FIG_STYLE)
figs.micrograph, figs.micrograph_crop = MICROGRAPH, MICROGRAPH_CROP
manifest = figs.generate_all(FIG_DIR, formats=FORMATS)


In [ ]:
# ---- Review the set, with its draft captions ---------------------------------------
from IPython.display import Image as IPyImage, Markdown

display(Markdown((FIG_DIR / "captions.md").read_text(encoding="utf-8")))

for f in manifest["figures"]:
    png = next((p for p in f["files"] if p.endswith(".png")), None)
    if png:
        display(Markdown(f"---\n**Figure {f['n']} — {f['title']}**"))
        display(IPyImage(filename=png, width=620))
    else:
        display(Markdown(f"---\n**Figure {f['n']} — {f['title']}**  \n"
                         f"_skipped: {f.get('skipped')}_"))

print(f"\nVector PDFs and 600 dpi PNGs: {FIG_DIR}")
print("Drop the PDFs straight into LaTeX; captions.md is a starting point, not a "
      "finished caption — check every number in it before submission.")


In [ ]:
# ============ INSPECT EVERY DOWNLOADED EXPORT FOR THIS RUN ============
# Reads local files only — no instrument, no network. Safe to re-run.
# Handles BOTH containers: the CSV family (<test>_<run>_export_Results.CSV +
# _Test<n>.CSV) and the Excel workbook (<test>_<run>_export.xlsx, whose
# "Results" sheet is the per-indent table and whose "Test <n>" sheets are the
# curves — one per indent, so the count follows the array).
import re
from pathlib import Path
import pandas as pd
from IPython.display import display
from nanoindentation_agents.instrument import results as R

RUN_ID       = globals().get("RUN_ID", "0806_160942")
DOWNLOAD_DIR = Path(globals().get("DOWNLOAD_DIR", REPO_ROOT / "notebooks" / "downloaded_data"))
SHOW_RAW     = False    # True -> print each Results table verbatim
SHOW_CURVES  = True     # True -> one summary row per indent curve

EXPORT_EXTS = {".csv", *R.EXCEL_EXTS}

pd.set_option("display.width", 200)


def run_exports() -> list:
    """One record per test of this run: {test, files, summary, curves}."""
    out = {}
    for p in sorted(DOWNLOAD_DIR.iterdir()):
        if not p.is_file() or p.suffix.lower() not in EXPORT_EXTS:
            continue
        m = re.match(rf"(.+?)_{RUN_ID}_export(?:_(.+))?$", p.stem, re.I)
        if not m:
            continue
        test, kind = m.group(1), (m.group(2) or "")
        rec = out.setdefault(test, {"test": test, "files": [], "fmt": set(),
                                    "summary": None, "curves": {}})
        rec["files"].append(p)
        rec["fmt"].add(p.suffix.lstrip(".").lower())
        try:
            if p.suffix.lower() in R.EXCEL_EXTS and not kind:      # the workbook
                rec["summary"], rec["curves"] = R.read_inview_workbook(p)
            elif kind.lower() == "results":
                rec["summary"] = R.read_inview_table(p)
            elif re.fullmatch(r"test[\s_-]*\d+", kind, re.I):
                rec["curves"][int(re.search(r"\d+", kind).group())] = \
                    R.read_inview_table(p)
        except Exception as exc:
            print(f"  !! {p.name}: {type(exc).__name__}: {exc}")
    return [out[k] for k in sorted(out)]


records = run_exports()

# ---- 1. what actually came down ---------------------------------------------------
if not records:
    print(f"No downloaded exports for run {RUN_ID} in {DOWNLOAD_DIR}")
else:
    print(f"=== FILES DOWNLOADED FOR RUN {RUN_ID} ===")
    display(pd.DataFrame([{
        "test": r["test"], "format": "/".join(sorted(r["fmt"])),
        "files": len(r["files"]),
        "kB": round(sum(p.stat().st_size for p in r["files"]) / 1024, 1),
        "indent rows": 0 if r["summary"] is None else len(r["summary"]),
        "curves": len(r["curves"]),
    } for r in records]).set_index("test"))

# ---- 2. the per-indent summary of every test, with the campaign's OWN verdict ------
rows = []
for r in records:
    data = r["summary"]
    if data is None or data.empty or "Test" not in data:
        continue
    data = data[pd.to_numeric(data["Test"], errors="coerce").notna()]
    ok, why = R.validate_rows(R.normalize_result_columns(data.copy()),
                              expected_indents=len(data))
    for _, row in data.iterrows():
        rows.append({
            "test": r["test"],
            **{c: pd.to_numeric(row.get(c), errors="coerce") for c in
               ("Pmax", "hmax", "Hardness", "Modulus", "Stiffness^2/Load")},
            "used by campaign": "yes" if ok else "NO",
            "why not": "" if ok else why.split(" -- ")[0][:70]})
if rows:
    print(f"\n=== RESULTS CONTENT ({len(rows)} indent row(s)) ===")
    display(pd.DataFrame(rows).round(3))

# ---- 3. the raw curves ------------------------------------------------------------
if SHOW_CURVES:
    crows = []
    for r in records:
        for n, curve in sorted(r["curves"].items()):
            df = R.normalize_result_columns(curve)
            g = lambda c: (pd.to_numeric(df[c], errors="coerce") if c in df
                           else pd.Series(dtype=float))
            crows.append({"test": r["test"], "indent": n, "points": len(df),
                          "depth max (nm)": round(float(g("Contact Depth (nm)").max()), 1),
                          "load max (mN)": round(float(g("Load (mN)").max()), 2),
                          "H min": round(float(g("Hardness (GPa)").min()), 2),
                          "H max": round(float(g("Hardness (GPa)").max()), 2),
                          "E max (GPa)": round(float(g("Modulus (GPa)").max()), 1)})
    if crows:
        print("\n=== PER-INDENT CURVES ===")
        display(pd.DataFrame(crows))

# ---- 4. the tables verbatim -------------------------------------------------------
if SHOW_RAW:
    for r in records:
        for p in r["files"]:
            print(f"\n----- {p.name} -----")
            if p.suffix.lower() == ".csv":
                print(p.read_text(encoding="utf-8", errors="replace").rstrip())
            else:                  # a workbook has no verbatim text — show the grids
                for sheet, grid in pd.read_excel(p, sheet_name=None, header=None).items():
                    print(f"[sheet: {sheet}]")
                    display(grid)
